In [82]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import sparse
import math  
import sklearn.metrics 
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Resumen

1. Crear la matriz de usuarios/películas
2. Solucionar problema escasez de datos
3. Crear la matriz de similitud
4. KN-Neighbours
5. Validación de los datos
4. Visualización de datos con Matplotlib

## 1. Crear la matriz usuarios/películas

In [2]:
df = pd.read_csv("BBDD_100K/ratings.csv")
df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
indice=list(df['userId'].unique())
columnas=list(df['movieId'].unique())
indice=sorted(indice)
columnas=sorted(columnas)
 
escasez=pd.pivot_table(data=df,values='rating',index='userId',columns='movieId')
escasez.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,4.0,5.0,3.0,5.0,4.0,4.0,3.0,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Solucionar problema escasez de datos

Para solucionar la escasez de datos voy a probar normalizando todos los datos de la tabla

1. Primero voy a restar la calificación promedio de cada usuario para centrarla alrededor de 0 \
    2.1 Esto es útil porque facilita las comparaciones entre usuarios y/o ítems, eliminando posibles sesgos debido a diferentes escalas de calificaciones o preferencias personales de los usuarios. 
2. Finalmente voy a convertir los NaN a 0

In [4]:
valoraciones_promedio = escasez.mean(axis = 1)
restar_promedio = escasez.sub(valoraciones_promedio , axis = 0)

In [5]:
restar_promedio = restar_promedio.fillna(0) #Rellenamos valores sobrantes por 0
usuario_pelicula = restar_promedio.copy()

In [6]:
usuario_pelicula.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.366379,0.000000,-0.366379,0.000000,0.000000,-0.366379,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.363636,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.000000,0.506369,1.506369,-0.493631,1.506369,0.506369,0.506369,-0.493631,0.0,-0.493631,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1.269737,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.000000,0.425532,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-1.574468,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Crear la matriz de similitud

1. Primero separaré al usuario para predecir sus calificaciones, en nuestro caso lo haremos con el usuario número 3
2. Después eliminaré la película que quiero predecir (en este caso y gracias al EDA utilizaré la película con mas valoraciones, la cual es Forrest Gump, con el id nº356)
3. Finalmente crearé el conjunto de usuarios que vieron esa película en específico 

In [52]:
id_usuario = 3
usuario_objetivo = usuario_pelicula.iloc[[id_usuario]]
print("Usuario Objetivo:")
usuario_objetivo.head()

Usuario Objetivo:


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [53]:
id_pelicula = 356
pelicula_objetivo = usuario_pelicula.drop( id_pelicula , axis =1)
print("Película Objetivo:")
pelicula_objetivo.head()

Película Objetivo:


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.366379,0.0,-0.366379,0.0,0.0,-0.366379,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.363636,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [190]:
usuarios_que_vieron_la_pelicula_con_nulls = usuario_pelicula[356]  # Selecciono todas las filas respecto a la película en cuestión
usuarios_que_vieron_la_pelicula = usuario_pelicula[usuarios_que_vieron_la_pelicula_con_nulls != 0] # Seleccionamos solo las filas con valores diferentes a 0, de entre los que vieron esa película
#Muestra de los usuarios que vieron la película


print("Shape de usuarios_que_vieron_la_pelicula_con_nulls:", usuarios_que_vieron_la_pelicula_con_nulls.shape)
print("Shape de usuarios_que_vieron_la_pelicula:", usuarios_que_vieron_la_pelicula.shape)
usuarios_que_vieron_la_pelicula[356].head(20)

Shape de usuarios_que_vieron_la_pelicula_con_nulls: (610,)
Shape de usuarios_que_vieron_la_pelicula: (329, 9724)


userId
1    -0.366379
6     1.506369
7     1.769737
8    -0.574468
10    0.221429
11    1.218750
14    0.604167
15    1.551852
16   -0.224490
17    0.790476
18    0.767928
19   -0.607397
21    1.239278
22    2.428571
24    0.850000
26   -0.238095
27    1.451852
28    0.979825
29    0.358025
33    1.211538
Name: 356, dtype: float64

## 4. K-NNeighbours

 algoritmo de KNN lo utilizaré con la libreria sklearn, la cual nos proporciona el uso de diferentes distancias, tales como: minkowski,manhattan , euclídea, jaccard, chebyshev, coseno... Lo que haré en esta sección es aplicar KNN y comparar las diferentes distáncias para evaluar cual es la mejor

 1. Distancia Minkowski
 2. Distancia Manhattan
 3. Distancia Eucídeana
 4. Distancia Jaccard
 5. Distancia Chebyshev
 6. Distancia Coseno

 Recordar que el resultado obtenido por parte del KNN está normalizado a rango [-1,1], por tanto hay que reescalarlo para poder ver cual sería en realidad

In [83]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial import distance_matrix
from scipy.spatial.distance import euclidean, pdist, squareform
from sklearn.model_selection import GridSearchCV, cross_val_score

Para cuadrar las dimensiones

In [68]:
usuarios_que_vieron_la_pelicula_con_nulls = usuarios_que_vieron_la_pelicula_con_nulls.loc[usuarios_que_vieron_la_pelicula.index]

In [69]:
min_rating = 0.5
max_rating = 5.0

def reescalar(res_distancia):
    prediccion_reescalada = ((res_distancia + 1) / 2) * (max_rating - min_rating) + min_rating
    print("Prediccion reescalada: " +str(prediccion_reescalada))

Similitud mediante distancia de minkowski

La p de la distancia Minkowski la hemos de poner a 3 o superior, a mayor cambio mayores oscilaciones tendrá la predicción, debido al énfasis que se le pone al cambio brusco de distancias. Esto lo hago ya que si no pongo p=3, por defecto se utiliza p=2, que es la que usa la distancia euclideana y por tanto darían exactamente lo mismo

In [89]:
usuario_minkowski = KNeighborsRegressor(metric='minkowski',p=3, n_neighbors=20)
usuario_minkowski.fit(usuarios_que_vieron_la_pelicula, usuarios_que_vieron_la_pelicula_con_nulls)
prediccion_usuario_minkoswki = usuario_minkowski.predict(usuario_objetivo)
print("Prediccion normalizada: " +str(prediccion_usuario_minkoswki))
reescalar(prediccion_usuario_minkoswki)

Prediccion normalizada: [0.43444098]
Prediccion reescalada: [3.72749222]


Similitud mediante distancia de Manhattan

In [92]:
usuario_manhattan = KNeighborsRegressor(metric='manhattan', n_neighbors=20)
usuario_manhattan.fit(usuarios_que_vieron_la_pelicula, usuarios_que_vieron_la_pelicula_con_nulls)
prediccion_usuario_manhattan = usuario_manhattan.predict(usuario_objetivo)
print("Prediccion normalizada: " +str(prediccion_usuario_manhattan))
reescalar(prediccion_usuario_manhattan)


Prediccion normalizada: [0.2882469]
Prediccion reescalada: [3.39855553]


Similitud mediante distancia de Euclideana

In [93]:
usuario_euclideana = KNeighborsRegressor(metric='euclidean', n_neighbors=20)
usuario_euclideana.fit(usuarios_que_vieron_la_pelicula, usuarios_que_vieron_la_pelicula_con_nulls)
prediccion_usuario_euclideana = usuario_euclideana.predict(usuario_objetivo)
print("Prediccion normalizada: " +str(prediccion_usuario_euclideana))
reescalar(prediccion_usuario_euclideana)

Prediccion normalizada: [0.35993502]
Prediccion reescalada: [3.55985379]


Similitud mediante distancia de Jaccard

In [94]:
usuario_jaccard = KNeighborsRegressor(metric='jaccard', n_neighbors=20)
usuario_jaccard.fit(usuarios_que_vieron_la_pelicula, usuarios_que_vieron_la_pelicula_con_nulls)
prediccion_usuario_jaccard = usuario_jaccard.predict(usuario_objetivo)
print("Prediccion normalizada: " +str(prediccion_usuario_jaccard))
reescalar(prediccion_usuario_jaccard)

Prediccion normalizada: [0.39211862]
Prediccion reescalada: [3.63226689]


Similitud mediante distancia de Chabyshev

In [97]:
usuario_chebyshev = KNeighborsRegressor(metric='chebyshev', n_neighbors=20)
usuario_chebyshev.fit(usuarios_que_vieron_la_pelicula, usuarios_que_vieron_la_pelicula_con_nulls)
prediccion_usuario_chebyshev = usuario_chebyshev.predict(usuario_objetivo)
print("Prediccion normalizada: " +str(prediccion_usuario_chebyshev))
reescalar(prediccion_usuario_chebyshev)

Prediccion normalizada: [0.41651415]
Prediccion reescalada: [3.68715685]


Similitud mediante distancia de Coseno

In [99]:
usuario_coseno = KNeighborsRegressor(metric='cosine', n_neighbors=20)
usuario_coseno.fit(usuarios_que_vieron_la_pelicula, usuarios_que_vieron_la_pelicula_con_nulls)
prediccion_usuario_coseno = usuario_coseno.predict(usuario_objetivo)
print("Prediccion normalizada: " +str(prediccion_usuario_coseno))
reescalar(prediccion_usuario_coseno)

Prediccion normalizada: [0.59663854]
Prediccion reescalada: [4.09243671]


Como nos interesa saber cual es la mejor k posible, voy a unificar en diversas funciones todo lo hecho hasta el momento, automatizando los resultados y de esta forma poder evaluar cualquier usuario y cualquier película siempre que queramos

In [192]:
def peliculas_del_usuario(usuario, matriz_escasez):
    peliculas_totales=0
    lista_peliculas=[]
    peliculas = matriz_escasez.columns
    for i in matriz_escasez.iloc[usuario]: #aqui lo que vamos a hacer es mirar que películas se pueden votar que no sean NaN del usuario que he seleciconado
        if not(np.isnan(i)) :
            lista_peliculas.append(peliculas[peliculas_totales])
        peliculas_totales=peliculas_totales+1

    valoraciones_originales = []
    for i in lista_peliculas:
        valoraciones_originales.append(escasez.iloc[usuario][i])

    print("ID's de películas votadas por el usuario: " + str(lista_peliculas[:10]) + "... (total: " + str(len(lista_peliculas)) + ")")
    print("Valoraciones de las primeras películas del usuario: " + str(valoraciones_originales[:10]) + "... (total: " + str(len(valoraciones_originales)) + ")")
    return lista_peliculas, valoraciones_originales

In [198]:

def similitud_distancias(lista_peliculas_usuario,pelicula_a_predecir,distancia_escogida,vecinos, minkowski=False):
    lista_similitud = []
    for i in lista_peliculas_usuario:
        usuario_objetivo = usuario_pelicula.iloc[[i]]
        usuarios_que_vieron_la_pelicula_con_nulls = usuario_pelicula[pelicula_a_predecir]  # Selecciono todas las filas respecto a la película en cuestión
        usuarios_que_vieron_la_pelicula = usuario_pelicula[usuarios_que_vieron_la_pelicula_con_nulls != 0] # Seleccionamos solo las filas con valores diferentes a 0, de entre los que vieron esa película
        if minkowski:
            knn = KNeighborsRegressor(metric= distancia_escogida ,p=3 , n_neighbors=vecinos)
        else:
            knn = KNeighborsRegressor(metric= distancia_escogida, n_neighbors=vecinos)
        knn.fit(usuarios_que_vieron_la_pelicula, usuarios_que_vieron_la_pelicula_con_nulls)
        prediccion_usuario = knn.predict(usuario_objetivo)
        lista_similitud.append(prediccion_usuario[0])
    return lista_similitud

In [194]:
def mejor_cantidad_de_vecinos(lista_peliculas, valoraciones_peliculas, pelicula_a_predecir,distancia_escogida,minkowski=False):
    mejor_k = None
    mejor_error = 99999
    resultados_vecinos = []
    for k in range (1,21): #escogemos un rango de 1 a 21 ya que mas de 21 seria demasiada influencia en base al desplazamiento
        similitud = similitud_distancias(lista_peliculas,pelicula_a_predecir,distancia_escogida,k,minkowski)
        evaluacion = sklearn.metrics.mean_squared_error(valoraciones_peliculas, similitud)
        resultados_vecinos.append(evaluacion)
        print("The evaluation at k = "+ str(i) + " equals to "+ str(evaluacion))
        if evaluacion < mejor_error:
            mejor_k = k
            mejor_error = evaluacion
    
    print("---------------------Final Best result at K => " + str(mejor_k)+ "-------------------------------")   

In [196]:

lista_peliculas, valoraciones_peliculas = peliculas_del_usuario(3,escasez)


ID's de películas votadas por el usuario: [21, 32, 45, 47, 52, 58, 106, 125, 126, 162]... (total: 216)
Valoraciones de las primeras películas del usuario: [3.0, 2.0, 3.0, 2.0, 3.0, 3.0, 4.0, 5.0, 1.0, 5.0]... (total: 216)


In [197]:
mejor_cantidad_de_vecinos(lista_peliculas, valoraciones_peliculas,356,"manhattan")

ValueError: Found input variables with inconsistent numbers of samples: [329, 610]